# Mangrove Pulse: reproduction and evidence review

This notebook checks whether the repository reproduces the frozen baseline and audit from the saved comparison input, then displays the three historical review cases.

Use Python 3 with the default CPU runtime and RAM. Run the five sections in order. Package installation requires internet; optional HTML maps request remote imagery and scripts. Allow about 1 GB for dependencies and temporary outputs; installation plus execution should take a few minutes (estimate, not a measured run).

Input: `Mangrove_Pulse_Repository_Draft.zip`. The repository contains the exact successful `Khor_Kalba_Date_Comparison_Results.zip` and all required code. Upload the repository ZIP only when section 1 asks.

Download `Mangrove_Pulse_Submission_Check.zip` after success or failure. If any section fails, continue to section 5 to preserve diagnostics. Do not retune thresholds or replace inputs to make checks pass.

The original assembled workflow passed cached reproduction on 2026-09-28 (see `evidence/verified_run`). Remote imagery loading was not checked. This documentation revision preserves all executable notebook cells and scientific execution logic; it has not been rerun in Colab. The workflow repeats the frozen northern-only threshold rule (must return 0.49), compares with saved results, and never treats GMW agreement as field accuracy. No acquisition, metadata query, new model or new truth labels are produced.

Completed work includes reproducible satellite/reference analysis, held-out evaluation, disagreement and coverage audit, and three historical Khor Kalba review cases. The C001 inspection brief is an example handoff with proposed evidence-acceptance rules.

The proposed integration transfers locations, evidence and unanswered questions to field inspection. A drone is a candidate for closer observations and exists only as a concept.

Future research should test amphibious mobility and sensing separately. Consider planting only after establishing ecological need and suitability, followed by separate intervention and survival trials. The review cases do not establish ecological conditions or operational value.


## 1. Load and verify the repository
Run this cell. Select the repository ZIP if prompted. A fresh session folder prevents stale error logs from appearing in a new run. Repeating this cell starts a new run.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import csv, hashlib, html, io, json, shutil, subprocess, sys, traceback, uuid, zipfile
RUN = Path('/content') / ('Mangrove_Pulse_Check_' + uuid.uuid4().hex[:10])
RUN.mkdir(parents=True)
ROOT = None
PACKAGE_OK = SETUP_OK = False
state = {'status':'loading_package','started_utc':datetime.now(timezone.utc).isoformat(),
         'imagery_loading':'unverified_in_this_run'}
def save_status():
    (RUN/'session_status.json').write_text(json.dumps(state,indent=2))
def fail(stage,exc):
    state.update(status='failed',stage=stage,error=str(exc))
    (RUN/'failure_log.txt').write_text(traceback.format_exc())
    save_status()
    print('STOPPED:',stage,str(exc),'— run section 5 to download diagnostics.')
save_status()
try:
    candidates=[Path.cwd(),Path('/content/mangrove-pulse')]
    ROOT=next((p for p in candidates if (p/'SHA256SUMS.json').is_file() and (p/'src/run_submission.py').is_file()),None)
    if ROOT is None:
        from google.colab import files
        print('Select ONE Mangrove_Pulse_Repository_Draft.zip (not an earlier result ZIP).')
        uploaded=files.upload()
        if len(uploaded)!=1: raise ValueError('Select exactly one repository ZIP.')
        payload=next(iter(uploaded.values()))
        destination=RUN/'package';destination.mkdir()
        with zipfile.ZipFile(io.BytesIO(payload)) as z:
            entries=z.infolist()
            if len({x.filename for x in entries})!=len(entries): raise ValueError('Duplicate ZIP members.')
            if sum(x.file_size for x in entries)>100_000_000: raise ValueError('Unexpectedly large package.')
            for member in entries:
                p=(destination/member.filename).resolve()
                if not p.is_relative_to(destination.resolve()): raise ValueError('Unsafe ZIP member path.')
                if ((member.external_attr>>16)&0o170000)==0o120000: raise ValueError('ZIP symlinks are not supported.')
            z.extractall(destination)
        matches=list(destination.glob('*/SHA256SUMS.json'))
        if len(matches)!=1: raise ValueError('Expected one mangrove-pulse repository folder.')
        ROOT=matches[0].parent
    manifest=json.loads((ROOT/'SHA256SUMS.json').read_text())
    for name,expected in manifest.items():
        p=(ROOT/name).resolve()
        if not p.is_relative_to(ROOT.resolve()) or not p.is_file(): raise ValueError('Missing/unsafe file: '+name)
        if hashlib.sha256(p.read_bytes()).hexdigest()!=expected: raise ValueError('File changed: '+name)
    (RUN/'package_manifest.json').write_text(json.dumps(manifest,indent=2))
    PACKAGE_OK=True;state['status']='package_verified';save_status()
    print('PACKAGE VERIFIED:',len(manifest),'files. Continue to section 2.')
except Exception as exc:
    fail('package',exc)


## 2. Install the recorded dependencies
Wait for SETUP READY. These pins match the original successful baseline/audit environment; installation on a fresh runtime still needs this check. On failure, download diagnostics rather than changing versions blindly.


In [ ]:
SETUP_OK=False
if not PACKAGE_OK:
    print('Package verification failed. Run section 5.')
else:
    try:
        with (RUN/'setup_log.txt').open('w') as log:
            process=subprocess.run([sys.executable,'-m','pip','install','--disable-pip-version-check',
                '--retries','0','--timeout','25','-r',str(ROOT/'requirements.txt')],
                stdout=log,stderr=subprocess.STDOUT,timeout=300)
        if process.returncode: raise RuntimeError('Dependency installation failed; see setup_log.txt.')
        SETUP_OK=True;state.pop('error',None);state.pop('stage',None)
        state['status']='setup_ready';save_status();print('SETUP READY. Continue to section 3.')
    except Exception as exc:
        fail('setup',exc)


## 3. Reproduce the frozen baseline and audit
This is the only geospatial execution in the notebook. It uses the cached small crops, repeats the original northern-only selection, and compares summaries, decoded predictions, split and every audit record against the saved outputs. Baseline and audit subprocesses each have a five-minute limit. No satellite network requests occur.

A failure is a reproducibility finding. Preserve its diagnostics rather than changing the threshold, geography or reference. Rerunning this cell creates a fresh output directory.


In [ ]:
if not SETUP_OK:
    print('Setup is incomplete. Run section 5.')
else:
    try:
        CURRENT=RUN/('reproduction_'+uuid.uuid4().hex[:8]);CURRENT.mkdir()
        state.pop('error',None);state.pop('stage',None)
        state.update(status='reproducing',latest_reproduction=CURRENT.name);save_status()
        with (RUN/'workflow_log.txt').open('w') as log:
            p=subprocess.run([sys.executable,str(ROOT/'src/run_submission.py'),
                '--root',str(ROOT),'--out',str(CURRENT)],stdout=log,stderr=subprocess.STDOUT,timeout=660)
        print((RUN/'workflow_log.txt').read_text())
        if p.returncode: raise RuntimeError('Reproduction failed. See workflow_log.txt and the reproduction folder.')
        state['status']='reproduction_matched';save_status()
        if (RUN/'failure_log.txt').exists(): (RUN/'failure_log.txt').unlink()
        print('Continue to the recorded evidence review in section 4.')
    except Exception as exc:
        fail('reproduction',exc)


## 4. Inspect the recorded evidence
The table and plots below are saved results from the earlier successful runs. Section 3 reports separately whether this run reproduced them. F1 measures agreement with GMW, not independent field accuracy.

The historical review used GE01 imagery from archive 2020-10-14/release 119, with native tile zoom 17. Capture date at the three confirmed points is **2018-10-05**, not the archive date. Source resolution: 1.2 m; stated positional accuracy: 8.47 m. Those values do not establish relative layer alignment or truth at every cell. The repository retains the metadata and case notes; imagery remains with its provider.

If a map is blank, record the loading problem; do not interpret it as imagery absence or scientific change. Previous screenshots supported qualitative review, not individual-cell labels. Do not reclassify cases or assign ecological urgency.

**Example handoff:** read `docs/C001_INSPECTION_BRIEF.md` alongside the saved audit panels. It records historical metadata and qualitative observations, then requests the evidence still needed. New observation fields remain unfilled. Present-day field observations cannot directly validate 2018 land cover.


In [ ]:
if not PACKAGE_OK:
    print('No verified saved evidence is available. Run section 5.')
else:
    try:
        from IPython.display import display, Markdown, Image, HTML
        saved=json.loads((ROOT/'examples/baseline/baseline_summary.json').read_text())
        lines=['### Previously recorded GMW agreement','| Date | Eligible | F1 | Excluded GMW+ |',
               '|---|---:|---:|---:|']
        for date,result in saved['held_out'].items():
            lines.append(f"| {date} | {result['evaluation_coverage']['total']} | {result['all_eligible_southern_cells']['f1']:.4f} | {result['evaluation_excluded']['mapped_mangrove']} |")
        display(Markdown('\n'.join(lines)))
        print('Current reproduction status:',state['status'])
        display(Image(filename=str(ROOT/'examples/audit/audit_overview.png')))
        cases=json.loads((ROOT/'examples/review/case_reviews.json').read_text())
        for case in cases:
            display(Markdown(f"### {case['case']} — {case['category']} ({case['cells']} cells)\n"
                +case['observation']+' '+case['interpretation']+'\n\nEvidence needed: '+case['next_evidence']))
            source=(ROOT/f"examples/review/{case['case']}_review.html").read_text()
            display(HTML('<iframe title="'+case['case']+' saved review" width="100%" height="500" '
                         'srcdoc="'+html.escape(source,quote=True)+'"></iframe>'))
        print('Map display was attempted; browser loading remains unverified until you inspect it.')
    except Exception as exc:
        (RUN/'viewer_failure_log.txt').write_text(traceback.format_exc())
        state['viewer_error']=str(exc);save_status()
        print('Viewer failed:',str(exc),'— computational results remain separate. Run section 5.')


## 5. Download the check package
Run after success or failure to download `Mangrove_Pulse_Submission_Check.zip`. Record whether all three maps loaded, with screenshots only if a map differs or fails. The ZIP contains this run's status, logs, comparisons, versions, reproduced results and package manifest; remote imagery pixels are not downloaded into it.

Stop here. This run does not verify field accuracy, operational benefit, current conditions, organiser requirements or the seven shared Colab links.


In [ ]:
from google.colab import files
if 'RUN' not in globals():
    raise RuntimeError('Run section 1 first so diagnostics have a session folder.')
state['exported_utc']=datetime.now(timezone.utc).isoformat();save_status()
checksums={str(p.relative_to(RUN)):hashlib.sha256(p.read_bytes()).hexdigest()
           for p in RUN.rglob('*') if p.is_file() and 'package' not in p.relative_to(RUN).parts
           and p.name!='output_sha256.json'}
(RUN/'output_sha256.json').write_text(json.dumps(checksums,indent=2))
archive=Path('/content/Mangrove_Pulse_Submission_Check.zip')
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for p in sorted(RUN.rglob('*')):
        if p.is_file() and 'package' not in p.relative_to(RUN).parts:
            z.write(p,p.relative_to(RUN))
print('Download:',archive,'| session status:',state['status'])
files.download(str(archive))
